# Preparing the Raw Data

The CSV files downloaded from the Kenneth French Data Library cannot be loaded directly using a normal `pandas.read_csv()` command.

Each downloaded file contains:

- Notes and descriptions above the monthly data
- More than one section in some files
- Annual returns or other information below the required monthly data
- Dates stored in `YYYYMM` format
- Missing-value codes such as `-99.99` and `-999`

This notebook identifies the required monthly section, removes the additional content and converts the data into a consistent format for the remaining notebooks.

## What the cleaning code does

For each of the eight raw files, the notebook:

1. Finds the required monthly data section.
2. Reads only rows with a six-digit monthly date.
3. Stops when the monthly section ends.
4. renames the first column as `date`.
5. Converts the date from `YYYYMM` to a Python date.
6. Converts the return columns to numeric values.
7. Replaces the Kenneth French missing-value codes with proper missing values.
8. Keeps observations from November 1990 to March 2011.
9. Saves the result in the `cleaned_data` folder.

The final checks confirm the number of rows and columns, sample period, missing values and duplicate dates.

## Manual alternative

The files can also be cleaned manually using Excel or another spreadsheet program.

For each file:

1. Open the raw CSV file.
2. Find the value-weighted monthly return section required for the analysis.
3. Delete all notes and descriptions above the column headings.
4. Delete annual returns and any other sections below the monthly data.
5. Keep only observations from November 1990 to March 2011.
6. Rename the first column as `date`.
7. Check for missing-value codes such as `-99.99` and `-999`.
8. Save the cleaned file as a new CSV in the `cleaned_data` folder.

This must be repeated for all eight files.

In [ ]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd

RAW_DIR = Path("raw_data")
CLEAN_DIR = Path("cleaned_data")

CLEAN_DIR.mkdir(exist_ok=True)

START_DATE = "1990-11-01"
END_DATE = "2011-03-01"

In [ ]:
def read_french_csv(path, section=None, required_columns=None):
    lines = path.read_text(encoding="utf-8-sig", errors="replace").splitlines()

    if section is not None:
        section_index = next(
            i for i, line in enumerate(lines)
            if section in line
        )
        header_index = next(
            i for i in range(section_index + 1, len(lines))
            if lines[i].strip()
        )
    else:
        header_index = next(
            i for i, line in enumerate(lines)
            if all(column in line for column in required_columns)
        )

    header = next(csv.reader([lines[header_index]]))
    header = [column.strip() for column in header]
    header[0] = "date"

    rows = []

    for line in lines[header_index + 1:]:
        row = next(csv.reader([line]))
        first_value = row[0].strip() if row else ""

        if re.fullmatch(r"\d{6}", first_value):
            rows.append([value.strip() for value in row])
        elif rows:
            break

    df = pd.DataFrame(rows, columns=header)

    df["date"] = pd.to_datetime(df["date"], format="%Y%m")

    for column in df.columns[1:]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.replace([-99.99, -999], np.nan)

    df = df[
        df["date"].between(START_DATE, END_DATE)
    ].reset_index(drop=True)

    return df

In [ ]:
files = {
    "developed_3_factors.csv": {
        "required_columns": ["Mkt-RF", "SMB", "HML", "RF"]
    },
    "japan_3_factors.csv": {
        "required_columns": ["Mkt-RF", "SMB", "HML", "RF"]
    },
    "developed_momentum.csv": {
        "required_columns": ["WML"]
    },
    "japan_momentum.csv": {
        "required_columns": ["WML"]
    },
    "developed_25_size_bm.csv": {
        "section": "Average Value Weighted Returns -- Monthly"
    },
    "japan_25_size_bm.csv": {
        "section": "Average Value Weighted Returns -- Monthly"
    },
    "developed_25_size_momentum.csv": {
        "section": "Average Value Weighted Returns -- Monthly"
    },
    "japan_25_size_momentum.csv": {
        "section": "Average Value Weighted Returns -- Monthly"
    },
}

In [ ]:
cleaned = {}

for filename, settings in files.items():
    df = read_french_csv(
        RAW_DIR / filename,
        section=settings.get("section"),
        required_columns=settings.get("required_columns"),
    )

    output_path = CLEAN_DIR / filename
    df.to_csv(output_path, index=False)

    cleaned[filename] = df

In [ ]:
summary = pd.DataFrame(
    {
        "file": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "start": df["date"].min(),
        "end": df["date"].max(),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_dates": int(df["date"].duplicated().sum()),
    }
    for filename, df in cleaned.items()
)

summary